In [ ]:
# prepare
import emap
import json

# for simplicity, the following examples all use this simple cost model
def simple_cost_model(type_: str, *ports) -> float:
    if type_ == "$dff":
        return len(ports[0]) * 1.0
    elif type_ in {"$muls", "$mulu"}:
        return len(ports[0]) * len(ports[1]) * 1.0
    elif type_ in {"$adds", "$addu", "$subs", "$subu"}:
        return min(len(ports[0]) + len(ports[1]), len(ports[2])) * 1.0
    elif type_.startswith("$"): # other types
        return len(ports[0]) * 1.0
    return 0.0  # blackboxes or tech cells

### Example: Multiplier with Synchronous Reset

A DSP configuration that has synchronous reset / enable / dynamic control signals is not supported by `Lakeroad` currently. However, you can still provide `Nextmap` with manual DSP mapping rules to handle these features.

In [ ]:
!yosys -q -p "read_verilog tests/multiplier_with_rst.v; proc; write_json multiplier_with_rst.json"

In [ ]:
dsp_rules = {
    "signed_mul_2_stage_26_17_48_bit_rst": {    # rule name
        "requirements": {                       # resource requirements
            "dsp48e2": 1                        # use one DSP48E2
        },
        "hidden_inputs": ["clk"],               # hidden input ports, e.g., clock
        "inputs": ["a", "b", "rst"],            # input ports
        "outputs": ["p"],                       # output ports
        # and a match pattern in SQL
        "match_sql": """
            SELECT sdff_a.d, sdff_b.d, sdff_p.rst, sdff_p.q
            FROM sdffs AS sdff_a JOIN sdffs AS sdff_b JOIN aby_cells AS mul JOIN sdffs AS sdff_p
            ON sdff_a.q = mul.a AND sdff_b.q = mul.b AND mul.y = sdff_p.d
            WHERE mul.type = '$muls'
                AND width_of(sdff_a.d) <= 26 AND width_of(sdff_b.d) <= 17 AND width_of(sdff_p.q) <= 48
                AND sdff_a.rst = sdff_b.rst AND sdff_b.rst = sdff_p.rst
                AND sdff_a.rst_val = 0 AND sdff_b.rst_val = 0 AND sdff_p.rst_val = 0
        """
        # note that DSP48E2 can only support reset to zero
    }
}

In [ ]:
TEST_NAME = "multiplier_with_rst"
SCHEMA_PATH = "emap/schema.sql"
netlist = emap.NetlistDB(SCHEMA_PATH)
with open(f"{TEST_NAME}.json", "r") as f:
    netlist.build_from_json(json.load(f)["modules"]["top"])

netlist.rebuild()

cnt = emap.rewrites.rewrite_sdff(netlist)   # rewrite $dff to $sdff
print(f"Applied {cnt} rewrites")

# techmapping
emap.rewrites.create_tech_tables(netlist, dsp_rules)
emap.rewrites.rewrite_tech(netlist, dsp_rules)

mod = emap.extracts.ilp.extract_techmap_with_limit(netlist, simple_cost_model, dsp_rules, {"dsp48e2": 1}, OutputFlag=False)
with open(f"{TEST_NAME}_extracted.json", "w") as f:
    json.dump({"creator": "nextmap", "modules": {"top": mod}}, f, indent=2)

### Example: Word-level Register Splitting

In a lot of cases, bitblasting word-level cells and registers into bit-level ones helps exploit more optimizations if you can afford the increased ILP solving time. `Nextmap` provides a series of rewrites to split word-level registers into bit-level registers. Here is an example of how to use it.

In [ ]:
!yosys -q -p "read_verilog tests/signed_reg.v; proc; write_json signed_reg.json"

In [ ]:
TEST_NAME = "signed_reg"
SCHEMA_PATH = "emap/schema.sql"
netlist = emap.NetlistDB(SCHEMA_PATH)
with open(f"{TEST_NAME}.json", "r") as f:
    netlist.build_from_json(json.load(f)["modules"]["top"])

netlist.rebuild()
dff_word_matches = emap.rewrites.ematch_word_dff(netlist)
cnt = emap.rewrites.apply_word_dff_split(netlist, dff_word_matches)
print(f"Applied {cnt} rewrites")
netlist.rebuild()

with open(f"{TEST_NAME}_after_split.json", "w") as f:
    json.dump(netlist.dump_tables(), f, indent=2)

mod = emap.extracts.ilp.extract_no_techmap(netlist, simple_cost_model, OutputFlag=False)

with open(f"{TEST_NAME}_extracted.json", "w") as f:
    json.dump({"creator": "nextmap", "modules": {"top": mod}}, f, indent=2)

### Example: Systolic Array (DSPs)

This is a large real world design. We recommend enabling `emapcc` to speed up the flow.

In [ ]:
!yosys -q -p "read_verilog tests/systolic.v; proc; opt_muxtree; opt_reduce; opt_merge; opt_share; opt_clean; opt_expr; write_json systolic.json"

In [ ]:
dsp_rules = {
    "signed_mul_1_stage_26_17_48_bit": {    # rule name
        "requirements": {                   # resource requirements
            "dsp48e2": 1                    # use one DSP48E2
        },
        "hidden_inputs": ["clk"],   # hidden input ports, e.g., clock
        "inputs": ["a", "b"],       # input ports
        "outputs": ["p"],           # output ports
        # and a match pattern in SQL
        "match_sql": """
            SELECT mul1.a, mul1.b, dff1.q
            FROM dffs AS dff1 JOIN aby_cells AS mul1
            ON dff1.d = mul1.y
            WHERE mul1.type = '$muls'
                AND width_of(mul1.a) <= 26 AND width_of(mul1.b) <= 17 AND width_of(dff1.q) <= 48
        """
    },
    "signed_muladd_1_stage_27_18_48_bit": {
        "requirements": {
            "dsp48e2": 1
        },
        "hidden_inputs": ["clk"],
        "inputs": ["a", "b", "c"],
        "outputs": ["p"],
        "match_sql": """
            SELECT mul1.a, mul1.b, add1.b, dff1.q
            FROM dffs AS dff1 JOIN aby_cells AS mul1 JOIN aby_cells AS add1
            ON dff1.d = add1.y AND mul1.y = add1.a
            WHERE mul1.type = '$muls' AND add1.type = '$adds'
                AND width_of(mul1.a) <= 27 AND width_of(mul1.b) <= 18 AND width_of(add1.b) <= 48 AND width_of(dff1.q) <= 48
        """
    }
}

In [ ]:
TEST_NAME = "systolic"
SCHEMA_PATH = "emap/schema.sql"
netlist = emap.NetlistDB(SCHEMA_PATH)
with open(f"{TEST_NAME}.json", "r") as f:
    netlist.build_from_json(json.load(f)["modules"]["systolic"])

netlist.rebuild()

unsigned_add_to_signed_matches = emap.rewrites.ematch_unsigned_add_to_signed(netlist)
cnt = emap.rewrites.apply_unsigned_add_to_signed(netlist, unsigned_add_to_signed_matches)
print(f"Applied {cnt} rewrites")
signed_arith_cells = emap.rewrites.select_aby_cell_by_type(netlist, ["$adds", "$muls"])
cnt = emap.rewrites.apply_signed_arith_input_trunc(netlist, signed_arith_cells)
print(f"Applied {cnt} rewrites")
netlist.rebuild()

cnt = 1
while cnt > 0:
    comm_matches = emap.rewrites.ematch_comm(netlist, ["$adds", "$addu", "$muls", "$mulu"])
    assoc_to_right_matches = emap.rewrites.ematch_assoc_to_right(netlist, ["$adds", "$addu", "$muls", "$mulu"])
    assoc_to_left_matches = emap.rewrites.ematch_assoc_to_left(netlist, ["$adds", "$addu", "$muls", "$mulu"])
    dff_forward_aby_cell_matches = emap.rewrites.ematch_dff_forward_aby_cell(netlist, ["$adds", "$addu", "$muls", "$mulu"])
    dff_backward_aby_cell_matches = emap.rewrites.ematch_dff_backward_aby_cell(netlist, ["$adds", "$addu", "$muls", "$mulu"])

    cnt = 0
    cnt += emap.rewrites.apply_comm(netlist, comm_matches)
    cnt += emap.rewrites.apply_assoc_to_right(netlist, assoc_to_right_matches)
    cnt += emap.rewrites.apply_assoc_to_left(netlist, assoc_to_left_matches)
    cnt += emap.rewrites.apply_dff_forward_aby_cell(netlist, dff_forward_aby_cell_matches)
    cnt += emap.rewrites.apply_dff_backward_aby_cell(netlist, dff_backward_aby_cell_matches)

    if cnt > 0:
        print(f"Applied {cnt} rewrites")
    else:
        print("No rewrites applied, stopping")
    netlist.rebuild()

# techmapping
emap.rewrites.create_tech_tables(netlist, dsp_rules)
emap.rewrites.rewrite_tech(netlist, dsp_rules)

mod = emap.extracts.ilp.extract_techmap_with_limit(netlist, simple_cost_model, dsp_rules, {"dsp48e2": 1024}, OutputFlag=False)
with open(f"{TEST_NAME}_extracted.json", "w") as f:
    json.dump({"creator": "nextmap", "modules": {"systolic": mod}}, f, indent=2)

### Example: Systolic Array

This time we try to map the same design with different technology libraries to show the flexibility of `Nextmap`. We apply the same set of general rewrites but two sets of technology mapping rules.

In [ ]:
!yosys -q -p "read_verilog tests/systolic_matmul.v; proc; memory_map; opt; write_json systolic_matmul.json"

### Techmapping on MLPs

In [ ]:
mlp_rules = {
    "pe_mac": {
        "requirements": {
            "pe_achronix": 1
        },
        "hidden_inputs": ["clk"],
        "inputs": ["a", "b", "rst"],
        "outputs": ["a_out", "b_out", "c"],
        "match_sql": """
            SELECT dff_a.d, dff_b.d, sdff_c.rst, dff_a.q, dff_b.q, add1.y
            FROM dffs AS dff_a JOIN dffs AS dff_b JOIN aby_cells AS mul1 JOIN sdffs AS sdff_c JOIN aby_cells AS add1
            ON dff_a.q = mul1.a AND dff_b.q = mul1.b AND mul1.y = add1.b AND sdff_c.q = add1.a AND sdff_c.d = add1.y
            WHERE mul1.type = '$muls' AND add1.type = '$adds'
                AND width_of(mul1.a) <= 16 AND width_of(mul1.b) <= 16 AND width_of(add1.b) <= 32
                AND width_of(add1.a) <= 32 AND width_of(add1.y) <= 32
        """
    },
    "2x2_mesh_mac": {
        "requirements": {
            "mlp_achronix": 1
        },
        "hidden_inputs": ["clk"],
        "inputs": [
            "a00", "a10",
            "b00", "b01",
            "rst00", "rst01", "rst10", "rst11"
        ],
        "outputs": [
            "a_out01", "a_out11",
            "b_out10", "b_out11",
            "c00", "c01", "c10", "c11"
        ],
        "match_sql": """
            SELECT pe00.a, pe10.a, pe00.b, pe01.b,
                pe00.rst, pe01.rst, pe10.rst, pe11.rst,
                pe01.a_out, pe11.a_out, pe10.b_out, pe11.b_out,
                pe00.c, pe01.c, pe10.c, pe11.c
            FROM tech_pe_mac AS pe00 JOIN tech_pe_mac AS pe01 JOIN tech_pe_mac AS pe10 JOIN tech_pe_mac AS pe11
            ON pe00.a_out = pe01.a AND pe00.b_out = pe10.b AND pe01.b_out = pe11.b AND pe10.a_out = pe11.a
        """
    }
}

In [ ]:
TEST_NAME = "systolic_matmul"
SCHEMA_PATH = "emap/schema.sql"
netlist = emap.NetlistDB(SCHEMA_PATH)
with open(f"{TEST_NAME}.json", "r") as f:
    netlist.build_from_json(json.load(f)["modules"]["top"])

netlist.rebuild()

cnt = 1
while cnt > 0:
    comm_matches = emap.rewrites.ematch_comm(netlist, ["$adds"])
    dff_forward_aby_cell_matches = emap.rewrites.ematch_dff_forward_aby_cell(netlist, ["$adds", "$muls"])
    dff_backward_aby_cell_matches = emap.rewrites.ematch_dff_backward_aby_cell(netlist, ["$adds", "$muls"])

    cnt = 0
    cnt += emap.rewrites.apply_comm(netlist, comm_matches)
    cnt += emap.rewrites.apply_dff_forward_aby_cell(netlist, dff_forward_aby_cell_matches)
    cnt += emap.rewrites.apply_dff_backward_aby_cell(netlist, dff_backward_aby_cell_matches)

    if cnt > 0:
        print(f"Applied {cnt} rewrites")
    else:
        print("No rewrites applied, stopping")
    netlist.rebuild()

cnt = emap.rewrites.rewrite_sdff(netlist) # rewrite $dff to $sdff
print(f"Applied {cnt} rewrites")
# techmapping
emap.rewrites.create_tech_tables(netlist, mlp_rules)
emap.rewrites.rewrite_tech(netlist, mlp_rules)

# with open(f"{TEST_NAME}_after_techmap.json", "w") as f:
#     json.dump(netlist.dump_tables(), f, indent=2)

mod = emap.extracts.ilp.extract_techmap_with_limit(netlist, simple_cost_model, mlp_rules, {"pe_achronix": 0, "mlp_achronix": 4}, OutputFlag=False)
with open(f"{TEST_NAME}_extracted.json", "w") as f:
    json.dump({"creator": "nextmap", "modules": {"systolic": mod}}, f, indent=2)

### Techmapping on DSPs

In [ ]:
dsp_rules = {
    "signed_muladd_1_stage_27_18_48_bit_with_ab_out": {
        "requirements": {
            "dsp48e2": 1
        },
        "hidden_inputs": ["clk"],
        "inputs": ["a", "b", "c"],
        "outputs": ["a_out", "b_out", "p"],
        "match_sql": """
            SELECT dff_a.d, dff_b.d, add1.a, dff_a.q, dff_b.q, dff_c.q
            FROM dffs AS dff_a JOIN dffs AS dff_b JOIN aby_cells AS mul1 JOIN dffs AS dff_c JOIN aby_cells AS add1
            ON mul1.a = dff_a.d AND mul1.b = dff_b.d AND mul1.y = add1.b AND add1.y = dff_c.d
            WHERE mul1.type = '$muls' AND add1.type = '$adds'
                AND width_of(mul1.a) <= 27 AND width_of(mul1.b) <= 18 AND width_of(add1.b) <= 48 AND width_of(dff_c.q) <= 48
        """
    }
}

In [ ]:
TEST_NAME = "systolic_matmul"
SCHEMA_PATH = "emap/schema.sql"
netlist = emap.NetlistDB(SCHEMA_PATH)
with open(f"{TEST_NAME}.json", "r") as f:
    netlist.build_from_json(json.load(f)["modules"]["top"])

netlist.rebuild()

cnt = 1
while cnt > 0:
    comm_matches = emap.rewrites.ematch_comm(netlist, ["$adds"])
    dff_forward_aby_cell_matches = emap.rewrites.ematch_dff_forward_aby_cell(netlist, ["$adds", "$muls"])
    dff_backward_aby_cell_matches = emap.rewrites.ematch_dff_backward_aby_cell(netlist, ["$adds", "$muls"])

    cnt = 0
    cnt += emap.rewrites.apply_comm(netlist, comm_matches)
    cnt += emap.rewrites.apply_dff_forward_aby_cell(netlist, dff_forward_aby_cell_matches)
    cnt += emap.rewrites.apply_dff_backward_aby_cell(netlist, dff_backward_aby_cell_matches)

    if cnt > 0:
        print(f"Applied {cnt} rewrites")
    else:
        print("No rewrites applied, stopping")
    netlist.rebuild()

cnt = emap.rewrites.rewrite_sdff(netlist) # rewrite $dff to $sdff
print(f"Applied {cnt} rewrites")
# techmapping
emap.rewrites.create_tech_tables(netlist, dsp_rules)
emap.rewrites.rewrite_tech(netlist, dsp_rules)

# with open(f"{TEST_NAME}_after_techmap.json", "w") as f:
#     json.dump(netlist.dump_tables(), f, indent=2)

mod = emap.extracts.ilp.extract_techmap_with_limit(netlist, simple_cost_model, dsp_rules, {"dsp48e2": 16}, OutputFlag=False)
with open(f"{TEST_NAME}_extracted.json", "w") as f:
    json.dump({"creator": "nextmap", "modules": {"systolic": mod}}, f, indent=2)

### Example: Bitblasting Helps!

In [ ]:
!yosys -q -p "read_verilog tests/redundant_adders.v; proc; write_json redundant_adders.json"

In [ ]:
TEST_NAME = "redundant_adders"
SCHEMA_PATH = "emap/schema.sql"
netlist = emap.NetlistDB(SCHEMA_PATH)
with open(f"{TEST_NAME}.json", "r") as f:
    netlist.build_from_json(json.load(f)["modules"]["top"])

netlist.rebuild()

cnt = 1
while cnt > 0:
    unsigned_add_matches = emap.rewrites.select_aby_cell_by_type(netlist, ["$addu"])
    cnt = emap.rewrites.apply_unsigned_add_bitblast(netlist, ((a, b, y) for _, a, b, y in unsigned_add_matches))
    if cnt > 0:
        print(f"Applied {cnt} rewrites")
    else:
        print("No rewrites applied, stopping")
    netlist.rebuild()

mod = emap.extracts.ilp.extract_no_techmap(netlist, simple_cost_model, OutputFlag=False)
with open(f"{TEST_NAME}_extracted.json", "w") as f:
    json.dump({"creator": "nextmap", "modules": {"top": mod}}, f, indent=2)